In [1]:
import sys
#CHANGED APPROACH TO WORK WITH MASSES INSTEAD OF AA to get lower runtime
# Please do not remove package declarations because these are used by the autograder.

MASSES = {
    "G": 57, "A": 71, "S": 87, "P": 97, "V": 99, "T": 101, "C": 103, "I": 113, "L": 113,
    "N": 114, "D": 115, "K": 128, "Q": 128, "E": 129, "M": 131, "H": 137, "F": 147,
    "R": 156, "Y": 163, "W": 186,
}

# Create a list of unique amino acid masses (this is the changed part)
UNIQUE_MASSES = sorted(set(MASSES.values()))

def Mass(peptide):
    return sum(peptide)

def Expand(Leaderboard,masses):
    newLB = set()
    for pep in Leaderboard:
        for mass in masses:
            newLB.add(pep + (mass,))
    return newLB

def LinearSpectrum(peptide):
    preMass = [0]
    for mass in peptide:
        preMass.append(preMass[-1] + mass)
    
    linear_spectrum = [0]
    for i in range(len(peptide)):
        for j in range(i + 1, len(peptide) + 1):
            
            linear_spectrum.append(preMass[j] - preMass[i])
    return sorted(linear_spectrum)

def CyclicSpectrum(peptide):
    n = len(peptide)
    preMass = [0]
    for mass in peptide:
        preMass.append(preMass[-1] + mass)
    
    cyclic_spectrum = [0]
    for i in range(n):
        for j in range(1, n):
            if i + j <= n:
                cyclic_spectrum.append(preMass[i + j] - preMass[i])
            else:
                cyclic_spectrum.append(preMass[n] - (preMass[i] - preMass[i + j - n]))
    
    cyclic_spectrum.append(preMass[n])  # full mass
    
    return sorted(cyclic_spectrum)

def LinearScore(pep_spec, spectrum):
    score = 0
    spectrum_copy = spectrum[:]
    for value in pep_spec:
        if value in spectrum_copy:
            score += 1
            spectrum_copy.remove(value)
    return score

def cyclopeptide_scoring(peptide, spectrum):
    cyclic_spec = CyclicSpectrum(peptide)
    return LinearScore(cyclic_spec, spectrum)

def Trim(Leaderboard, spectrum, n):
    if not Leaderboard:
        return set()
    scores = []
    for pep in Leaderboard:
        pep_spec = LinearSpectrum(pep)
        score = LinearScore(pep_spec, spectrum)
        # print(pep,pep_spec,score)

        scores.append((score, pep))
    scores.sort(reverse=True) 
    # print(scores)
    # print(scores,n)
    if len(scores) <= n:
        return set([pep for score,pep in scores])
    
    # to get the scores in order but I need to now take the ties also when there are more than n peps
    nth_score = scores[n - 1][0]
    return set([pep for score, pep in scores if score >= nth_score])

def leaderboard_cyclopeptide_sequencing(spectrum: list[int], n: int, masses) -> list[int]:

    Leaderboard = {()}  
    LeaderPeptide = ()
    LeaderScore = 0
    parentmass = spectrum[-1]
    
    while Leaderboard:
        Leaderboard = Expand(Leaderboard,masses)  ### modify expand to use it in this case to use only the best_matches
        
        pep_to_remove = set()
        # print(Leaderboard)
        for Peptide in Leaderboard:
            peptide_mass = Mass(Peptide)
            # Complete peptide 
            if peptide_mass == parentmass:
                
                PeptideScore = cyclopeptide_scoring(Peptide, spectrum)

                if PeptideScore > LeaderScore:
                    LeaderPeptide = Peptide

                    LeaderScore = PeptideScore
                    # if Peptide==(160,170) or Peptide==(170,102,58):
                        # print(LeaderPeptide,Peptide)
                pep_to_remove.add(Peptide)

            #if it is larger (dont want to expand anymore)
            elif peptide_mass > parentmass:
                pep_to_remove.add(Peptide)
        
        # using pep to remove to avoid mutations like in the previous case
        Leaderboard -= pep_to_remove
        
        # Trim to top n and expand (restart while loop)
        Leaderboard = Trim(Leaderboard, spectrum, n)
    # print(LeaderPeptide)
    return list(LeaderPeptide)
# Insert your spectral_convolution function here, along with any subroutines you need
def spectral_convolution(spectrum: list[int]) -> list[int]:
    output=[]
    for i in range(len(spectrum)-1):
        for j in range(i,len(spectrum)):
            #print(spectrum[j],spectrum[i])
            value = abs(spectrum[j]-spectrum[i])
            if value>=57 and value < 200: 
                             
                output+=[abs(spectrum[j]-spectrum[i])]
    output = [x for x in output if x != 0]
    # print(output)
    return output 
# Please do not remove package declarations because these are used by the autograder.

def best_M(convolution,m):
    best_m = dict()
    for wt in convolution:
        best_m[wt]=best_m.get(wt,0)+1
        
    # print(best_m)
    result = []

    for k,v in best_m.items():
        result.append((v,k))
    
    result.sort(reverse=True)
    # print(result)   
    if len(result) < m:
        return set([mass for freq,mass in result])
    # print(result,m)
    # print(result[m - 1])
    mth = result[m - 1][0]
    # print(mth)
    # print(result)
    return set([mass for freq,mass in result if freq >= mth])
    

# Insert your convolution_cyclopeptide_sequencing function here, along with any subroutines you need
def convolution_cyclopeptide_sequencing(spectrum: list[int], m: int, n: int) -> list[int]:
    spectrum=sorted(spectrum)

   
    conv = spectral_convolution(spectrum)
    matches = best_M(convolution=conv,m=m)
    # print(matches)
    return leaderboard_cyclopeptide_sequencing(spectrum,n, masses=matches)






In [ ]:
# Code Challenge: Implement ConvolutionCyclopeptideSequencing().

# Input: A collection of (possibly repeated) integers Spectrum, an integer M, and an integer N.
# Output: A cyclic peptide LeaderPeptide with amino acids taken only from the top M elements (and ties) of the convolution of Spectrum that fall between 57 and 200, and where the size of Leaderboard is restricted to the top N (and ties).
# Sample Input:

# 57 57 71 99 129 137 170 186 194 208 228 265 285 299 307 323 356 364 394 422 493
# 20
# 60
# Sample Output:

# 99-71-137-57-57-72